In [10]:
import numpy as np
import pandas as pd


class UserBasedCF:

    def __init__(
        self,
        user_item_matrix,
        top_k=20,
        min_common=10
    ):

        self.user_item = user_item_matrix

        self.top_k = top_k
        self.min_common = min_common

        # ---------- NumPy matrix ----------

        self.matrix = self.user_item.to_numpy()

        self.user_to_idx = {
            user: i
            for i, user in enumerate(self.user_item.index)
        }

        self.movie_to_idx = {
            movie: j
            for j, movie in enumerate(self.user_item.columns)
        }

        # ---------- User Means ----------

        self.user_means = self.user_item.mean(axis=1)
        self.user_mean_array = self.user_means.to_numpy()

        # ---------- Neighbor Cache ----------

        self.neighbor_cache = {}

    # ==========================================================
    # Pearson Similarity
    # ==========================================================

    def pearson_similarity(
        self,
        user1,
        user2
    ):

        common = user1.notna() & user2.notna()

        common_count = common.sum()

        if common_count < self.min_common:
            return 0.0, common_count

        u1 = user1[common]
        u2 = user2[common]

        u1 = u1 - u1.mean()
        u2 = u2 - u2.mean()

        norm1 = np.linalg.norm(u1)
        norm2 = np.linalg.norm(u2)

        if norm1 == 0 or norm2 == 0:
            return 0.0, common_count

        similarity = np.dot(u1, u2) / (norm1 * norm2)

        return similarity, common_count

    # ==========================================================
    # Find Neighbors
    # ==========================================================

    def get_neighbors(
        self,
        target_user,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        if target_user in self.neighbor_cache:
            return self.neighbor_cache[target_user]

        target = self.user_item.loc[target_user]

        neighbors = []

        for other_user in self.user_item.index:

            if other_user == target_user:
                continue

            similarity, common = self.pearson_similarity(
                target,
                self.user_item.loc[other_user]
            )

            if similarity <= 0:
                continue

            neighbors.append(
                (
                    other_user,
                    similarity,
                    common
                )
            )

        neighbors.sort(
            key=lambda x: x[1],
            reverse=True
        )

        neighbors = neighbors[:top_k]

        self.neighbor_cache[target_user] = neighbors

        return neighbors

    # ==========================================================
    # Candidate Movies
    # ==========================================================

    def get_candidate_movies(
        self,
        target_user,
        top_k=None
    ):

        neighbors = self.get_neighbors(
            target_user,
            top_k
        )

        watched_movies = set(
            self.user_item.loc[target_user]
            .dropna()
            .index
        )

        candidate_movies = set()

        for neighbor_id, similarity, common in neighbors:

            neighbor_movies = (
                self.user_item
                .loc[neighbor_id]
                .dropna()
                .index
            )

            candidate_movies.update(
                neighbor_movies
            )

        candidate_movies -= watched_movies

        return list(candidate_movies)
    
    # ==========================================================
    # Predict Rating
    # ==========================================================

    def predict_rating(
        self,
        target_user,
        target_movie,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        # Movie not present in training
        if target_movie not in self.movie_to_idx:
            return None

        neighbors = self.get_neighbors(
            target_user,
            top_k
        )

        target_idx = self.user_to_idx[target_user]
        movie_idx = self.movie_to_idx[target_movie]

        target_mean = self.user_mean_array[target_idx]

        numerator = 0.0
        denominator = 0.0

        for neighbor_id, similarity, common in neighbors:

            neighbor_idx = self.user_to_idx[neighbor_id]

            rating = self.matrix[
                neighbor_idx,
                movie_idx
            ]

            if np.isnan(rating):
                continue

            neighbor_mean = self.user_mean_array[
                neighbor_idx
            ]

            numerator += similarity * (
                rating - neighbor_mean
            )

            denominator += similarity

        if denominator == 0:
            return None

        prediction = target_mean + (
            numerator / denominator
        )

        prediction = np.clip(
            prediction,
            1,
            5
        )

        return float(prediction)


    # ==========================================================
    # Recommend Movies
    # ==========================================================

    def recommend(
        self,
        target_user,
        top_n=10,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        recommendations = []

        candidate_movies = self.get_candidate_movies(
            target_user,
            top_k
        )

        for movie in candidate_movies:

            prediction = self.predict_rating(
                target_user,
                movie,
                top_k
            )

            if prediction is None:
                continue

            recommendations.append(
                (
                    movie,
                    prediction
                )
            )

        recommendations.sort(
            key=lambda x: x[1],
            reverse=True
        )

        recommendations = recommendations[:top_n]

        return pd.DataFrame(
            recommendations,
            columns=[
                "movie_id",
                "predicted_rating"
            ]
        )


    # ==========================================================
    # Evaluate
    # ==========================================================

    def evaluate(
        self,
        test_df
    ):

        predictions = []

        squared_errors = []

        absolute_errors = []

        for row in test_df.itertuples(index=False):

            user = row.user_id
            movie = row.movie_id
            actual = row.rating

            predicted = self.predict_rating(
                user,
                movie
            )

            if predicted is None:
                continue

            error = abs(
                actual - predicted
            )

            predictions.append(
                (
                    user,
                    movie,
                    actual,
                    predicted,
                    error
                )
            )

            absolute_errors.append(
                error
            )

            squared_errors.append(
                error ** 2
            )

        prediction_df = pd.DataFrame(
            predictions,
            columns=[
                "user_id",
                "movie_id",
                "actual",
                "predicted",
                "error"
            ]
        )

        mae = np.mean(
            absolute_errors
        )

        rmse = np.sqrt(
            np.mean(
                squared_errors
            )
        )

        return (
            prediction_df,
            rmse,
            mae
        )

In [11]:
train = pd.read_csv(
    "data/u1.base",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

test = pd.read_csv(
    "data/u1.test",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [12]:
train_matrix = train.pivot(
    index="user_id",
    columns="movie_id",
    values="rating"
)

In [13]:
cf = UserBasedCF(
    train_matrix,
    top_k=20,
    min_common=10
)

In [14]:
cf.get_neighbors(1)

[(69, np.float64(0.8448192338623107), np.int64(14)),
 (329, np.float64(0.8416650799680397), np.int64(11)),
 (320, np.float64(0.7957040966440955), np.int64(16)),
 (115, np.float64(0.7863336509949342), np.int64(12)),
 (21, np.float64(0.7690614556304467), np.int64(12)),
 (526, np.float64(0.6826021514709876), np.int64(11)),
 (22, np.float64(0.6780500082363685), np.int64(17)),
 (413, np.float64(0.6714450198148366), np.int64(11)),
 (197, np.float64(0.6648959685418467), np.int64(10)),
 (291, np.float64(0.6642456559897918), np.int64(32)),
 (517, np.float64(0.6442227924869779), np.int64(10)),
 (294, np.float64(0.6421554612114082), np.int64(14)),
 (246, np.float64(0.6259701640216627), np.int64(24)),
 (623, np.float64(0.6239181323861149), np.int64(13)),
 (540, np.float64(0.6161469221192538), np.int64(19)),
 (826, np.float64(0.6077192928383598), np.int64(25)),
 (82, np.float64(0.6015633408367517), np.int64(21)),
 (275, np.float64(0.6000192009216492), np.int64(11)),
 (890, np.float64(0.595189895870

In [15]:
cf.predict_rating(
    target_user=1,
    target_movie=300
)

3.801693681033199

In [16]:
import time
start = time.time()
recommendations = cf.recommend(
    target_user=1,
    top_n=10
)
print("Time taken for recommendations: ", time.time() - start)
recommendations

Time taken for recommendations:  0.017333984375


,movie_id,predicted_rating
0,12,5.0
1,201,5.0
2,303,5.0
3,315,5.0
4,330,5.0
5,347,5.0
6,433,5.0
7,483,5.0
8,509,5.0
9,616,5.0


In [ ]:

start = time.time()
prediction_df, rmse, mae = cf.evaluate(test)
print("Time taken for evaluation: ", time.time() - start)

print("MAE :", mae)
print("RMSE:", rmse)

prediction_df.head()

In [ ]:
import pstats
import cProfile

profiler = cProfile.Profile()

profiler.enable()

cf.evaluate(test)

profiler.disable()

stats = pstats.Stats(profiler)
stats.sort_stats("cumtime")
stats.print_stats(20)